# Indicators Admin Notebook

Manage indicator nodes: **Goal**, **SuccessIndicator**.

Goals are measurable objectives that an organization aims to achieve to enhance accessibility. Each goal is supported by success indicators, which represent specific metrics or benchmarks used to measure progress toward achieving that goal.

## Connection Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))

from app.database.graph_schema import set_connection
set_connection()

import pandas as pd
from datetime import date, datetime
pd.set_option('display.max_colwidth', 80)
print('Connection established.')

---
## READ Operations

### List All Goals

In [ ]:
from app.database.graph_schema import Goal

goals = Goal.nodes.all()
if goals:
    df = pd.DataFrame([g.serialize() for g in goals])
    display(df)
else:
    print('No Goal nodes found.')

### List All Success Indicators

In [ ]:
from app.database.graph_schema import SuccessIndicator

indicators = SuccessIndicator.nodes.all()
if indicators:
    df = pd.DataFrame([si.serialize() for si in indicators])
    display(df)
else:
    print('No SuccessIndicator nodes found.')

### List Goals with Their Success Indicators

In [ ]:
from app.database.graph_schema import Goal

goals = Goal.nodes.all()
for goal in goals:
    indicators = goal.supporting_success_indicators.all()
    print(f'\n=== Goal #{goal.goal_number}: {goal.name} (removed={goal.removed}) ===')
    if indicators:
        df = pd.DataFrame([si.serialize() for si in indicators])
        display(df)
    else:
        print('  No success indicators.')

### List Goals by Working Group

Shows which goals belong to each ATIWorkingGroup via the `responsible_for` relationship.

In [ ]:
from app.database.graph_schema import ATIWorkingGroup

wgs = ATIWorkingGroup.nodes.all()
for wg in wgs:
    goals = wg.responsible_for.all()
    print(f'\n=== {wg.name} ({len(goals)} goals) ===')
    if goals:
        df = pd.DataFrame([g.serialize() for g in goals])
        display(df)
    else:
        print('  No goals assigned.')

### Fetch Success Indicators for Working Group by Academic Year

Returns a nested JSON structure of working groups -> goals -> success indicators -> year success evidence for the specified academic year. Uses APOC for aggregation.

In [ ]:
from app.database.queries.indicators.read import fetch_success_indicators_for_working_group
import json

academic_year = '2024-2025'  # <-- change this

try:
    results = fetch_success_indicators_for_working_group(academic_year)
    if results:
        data = json.loads(results[0][0])
        for wg in data:
            print(f'\n=== {wg.get("name", "Unknown")} ===')
            for goal in wg.get('goals', []):
                print(f'  Goal #{goal.get("goal_number")}: {goal.get("name")}')
                for si in goal.get('successIndicators', []):
                    yse_count = len(si.get('yearSuccessIndicators', []))
                    print(f'    SI {si.get("composite_key")}: {si.get("success_indicator", "")[:60]}... ({yse_count} YSE)')
    else:
        print(f'No results for {academic_year}.')
except Exception as e:
    print(f'Error: {e}')

### Find a Specific Success Indicator by Composite Key

Composite key format: `goal_number.indicator_number-wg_abbreviation` (e.g., `1.1-web`, `2.3-pro`, `1.2-ins`)

In [ ]:
from app.database.graph_schema import SuccessIndicator

composite_key = '1.1-web'  # <-- change this

try:
    si = SuccessIndicator.nodes.get(composite_key=composite_key)
    print(f'Found: {si.composite_key}')
    print(f'  Number: {si.number}')
    print(f'  Text: {si.success_indicator}')
    print(f'  Removed: {si.removed}')
    print(f'  Date Added: {si.date_added}')
    print(f'  unique_id: {si.unique_id}')
except SuccessIndicator.DoesNotExist:
    print(f'No SuccessIndicator with composite_key "{composite_key}".')

---
## CREATE Operations

### Add a Goal

Parameters:
- `goal` (str, required) - The full goal text
- `goal_number` (int, required) - The goal number (must be unique within working group)
- `name` (str, required) - Short name for the goal
- `removed` (bool) - Whether the goal is removed
- `working_group` (str, required) - One of: `"web"`, `"ins"`, `"pro"`

In [ ]:
from app.database.queries.indicators.create import add_goal

# add_goal(
#     goal='Full goal text describing the accessibility objective',
#     goal_number=1,
#     name='Short Goal Name',
#     removed=False,
#     working_group='web'  # web, ins, or pro
# )

### Add a Success Indicator

Parameters:
- `number` (int, required) - The indicator number within its goal
- `goal_number` (int, required) - The goal this indicator belongs to
- `sub_committee` (str, required) - One of: `"web"`, `"ins"`, `"pro"`
- `success_indicator_text` (str, required) - The full text of the success indicator
- `date_added` (str, optional) - Date in YYYY-MM-DD format (defaults to today)
- `removed` (bool, default False)

The composite key is auto-generated as `goal_number.number-wg_abbreviation` (e.g., `1.1-web`).

In [ ]:
from app.database.queries.indicators.create import create_success_indicator

# create_success_indicator(
#     number=1,
#     goal_number=1,
#     sub_committee='web',  # web, ins, or pro
#     success_indicator_text='Description of what success looks like for this indicator',
#     date_added='2024-08-01',  # optional, defaults to today
#     removed=False
# )

---
## UPDATE Operations

### Update a Goal

Fetch by unique_id or goal_number, modify properties, and save.

In [ ]:
from app.database.graph_schema import Goal

uid = 'PASTE_UNIQUE_ID_HERE'  # <-- change this

# goal = Goal.nodes.get(unique_id=uid)
# goal.name = 'Updated Goal Name'
# goal.description = 'Updated description'
# goal.removed = False
# goal.save()
# print(f'Updated Goal #{goal.goal_number}: {goal.name}')

### Set Removed Status for a Success Indicator

Parameters:
- `composite_key` (str) - e.g. `"1.1-web"`
- `removed` (bool) - True to mark as removed, False to restore

In [ ]:
from app.database.queries.indicators.update import set_removed_status_for_success_indicator

# set_removed_status_for_success_indicator(
#     composite_key='1.1-web',
#     removed=True
# )

### Update a Success Indicator

Fetch by composite_key, modify properties, and save.

In [ ]:
from app.database.graph_schema import SuccessIndicator

composite_key = '1.1-web'  # <-- change this

# si = SuccessIndicator.nodes.get(composite_key=composite_key)
# si.success_indicator = 'Updated indicator text'
# si.save()
# print(f'Updated: {si.composite_key}')

---
## DELETE Operations

No dedicated delete functions exist for indicators. Use neomodel directly with caution, as deleting a Goal or SuccessIndicator will affect all connected relationships.

In [ ]:
from app.database.graph_schema import Goal, SuccessIndicator
from neomodel import db

# Delete a SuccessIndicator by composite_key (DETACH DELETE removes all relationships):
# composite_key = '1.1-web'
# si = SuccessIndicator.nodes.get(composite_key=composite_key)
# db.cypher_query('MATCH (n) WHERE elementId(n) = $eid DETACH DELETE n', {'eid': si.element_id})
# print(f'Deleted SuccessIndicator: {composite_key}')

# Delete a Goal by unique_id (DETACH DELETE removes all relationships):
# uid = 'PASTE_UNIQUE_ID_HERE'
# goal = Goal.nodes.get(unique_id=uid)
# db.cypher_query('MATCH (n) WHERE elementId(n) = $eid DETACH DELETE n', {'eid': goal.element_id})
# print(f'Deleted Goal #{goal.goal_number}')

print('Uncomment the delete operation you need and run again.')

---
## RELATIONSHIP Management

### View All Relationships for a Goal

In [ ]:
from app.database.graph_schema import Goal

goal = Goal.nodes.first()  # <-- change lookup as needed
if goal:
    print(f'=== Goal #{goal.goal_number}: {goal.name} ===')
    print(f'  Goal text: {goal.goal}')
    print(f'  Removed: {goal.removed}')
    
    indicators = goal.supporting_success_indicators.all()
    print(f'\nSuccess Indicators ({len(indicators)}):')
    for si in indicators:
        print(f'  - {si.composite_key}: {si.success_indicator}')
    
    plans = goal.furthered_by.all()
    print(f'\nFurthered by Plans ({len(plans)}):')
    for p in plans:
        print(f'  - {p.name}')
    
    accomplishments = goal.advanced_by.all()
    print(f'\nAdvanced by Accomplishments ({len(accomplishments)}):')
    for a in accomplishments:
        print(f'  - {a.name}')
    
    statuses = goal.status.all()
    print(f'\nStatus Levels ({len(statuses)}):')
    for s in statuses:
        print(f'  - {s.status_level} (value: {s.status_value})')
else:
    print('No Goal nodes found.')

### View All Relationships for a Success Indicator

In [ ]:
from app.database.graph_schema import SuccessIndicator

si = SuccessIndicator.nodes.first()  # <-- change lookup as needed
if si:
    print(f'=== {si.composite_key} ===')
    print(f'  Text: {si.success_indicator}')
    print(f'  Removed: {si.removed}')
    
    notes = si.notes.all()
    print(f'\nNotes ({len(notes)}):')
    for n in notes:
        print(f'  - {n.name}')
    
    plans = si.directed_plans.all()
    print(f'\nDirected Plans ({len(plans)}):')
    for p in plans:
        print(f'  - {p.name}')
    
    processes = si.directed_processes.all()
    print(f'\nDirected Processes ({len(processes)}):')
    for p in processes:
        print(f'  - {p.title}')
    
    projects = si.directed_projects.all()
    print(f'\nDirected Projects ({len(projects)}):')
    for p in projects:
        print(f'  - {p.title}')
    
    procedures = si.directed_procedures.all()
    print(f'\nDirected Procedures ({len(procedures)}):')
    for p in procedures:
        print(f'  - {p.title}')
    
    services = si.directed_services.all()
    print(f'\nDirected Services ({len(services)}):')
    for s in services:
        print(f'  - {s.title}')
else:
    print('No SuccessIndicator nodes found.')

### Connect a Note to a Success Indicator

In [ ]:
from app.database.graph_schema import SuccessIndicator, Note

# si = SuccessIndicator.nodes.get(composite_key='1.1-web')  # <-- change
# note = Note.nodes.get(unique_id='PASTE_NOTE_ID')  # <-- change

# if not si.notes.is_connected(note):
#     si.notes.connect(note)
#     print(f'Connected note "{note.name}" to {si.composite_key}')
# else:
#     print('Already connected.')

### Disconnect a Note from a Success Indicator

In [ ]:
from app.database.graph_schema import SuccessIndicator, Note

# si = SuccessIndicator.nodes.get(composite_key='1.1-web')  # <-- change
# note = Note.nodes.get(unique_id='PASTE_NOTE_ID')  # <-- change

# if si.notes.is_connected(note):
#     si.notes.disconnect(note)
#     print(f'Disconnected note "{note.name}" from {si.composite_key}')
# else:
#     print('Not connected.')